# Day 2 | Lab 2.3: Advanced Prompting — Zero-Shot, Few-Shot, CoT, ToT, CoVe

**Duration:** ~1.5 hours

**Scenario.** Retail + corporate banking (customer support, transaction categorization, structured intake, loan assessment, debt management, RBI guideline summarization, stock analysis)

**Learning Objectives.** By the end of this lab, you will be able to:
1. Apply **zero-shot** and **few-shot** prompting to common banking tasks and choose between them.
2. Use **Chain-of-Thought (CoT)** to make multi-step reasoning auditable.
3. Apply a **Tree-of-Thought (ToT)-style** prompt to explore multiple hypotheses before deciding.
4. Use **Chain-of-Verification (CoVe)** to reduce factual hallucinations on document summarization.
5. Decide which technique to use given a task's complexity, stakes, and latency budget.

**Tools.** LangChain v1 · `langchain-openai` · `gpt-4.1-mini` (CoT/ToT) · `gpt-4o-mini` (CoVe).

· Created by Prashant Sahu · [LinkedIn](https://www.linkedin.com/in/prashantksahu/)*

---


In [ ]:
# Required packages for this lab — already installed in your local venv.
# To install standalone, uncomment the line(s) below:
# !pip install -q 'langchain>=1.0' 'langchain-core>=1.0' 'langchain-openai>=1.0' 'langchain-community>=0.3' python-dotenv


In [1]:
import os

# Local-venv pattern: load from .env if python-dotenv is available, otherwise rely on
# environment variables already set in your shell or venv activation script.
try:
    from dotenv import load_dotenv
    load_dotenv("..\\.env")
except ImportError:
    pass

for key in ['OPENAI_API_KEY']:
    status = '✅ Loaded' if os.environ.get(key) else '❌ MISSING'
    print(f'{key}: {status}')


OPENAI_API_KEY: ✅ Loaded


In [23]:
import os
from IPython.display import display, Image, Markdown
# from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Load environment variables
# load_dotenv()

# Initialize the model (we use a slightly higher temperature for more creative/reasoning tasks)
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0.3)

# Helper function to print results neatly
def execute_prompt(prompt_template, title, **kwargs):
    """
    Executes a given prompt template with context variables and prints the output.
    """
    print(f"--- {title} ---")

    # Create the prompt using the template and input variables
    prompt = ChatPromptTemplate.from_template(prompt_template)

    # Create the chain
    chain = prompt | llm | StrOutputParser()

    # Invoke the chain with the provided context
    response = chain.invoke(kwargs)

    print("PROMPT:")
    print(prompt.format(**kwargs))
    print("\n" + "="*20 + " RESPONSE " + "="*20 + "\n")
    # print(response)
    display(Markdown(response))
    print("\n" + "-"*50 + "\n")

    return response

-----

## **1. Zero-Shot Prompting: Instructing Without Examples**

### **Background**

**Zero-shot prompting** is the most fundamental technique. It involves asking the model to perform a task for which it has received no specific, in-prompt examples. The model relies entirely on its pre-existing knowledge to understand the instruction and generate a response.

  * **How it works:** You give a direct command. For example, "Translate this text to French." The model has seen countless translation tasks during its training and knows what to do.
  * **Strengths:** Simple, fast, and effective for straightforward tasks that fall within the model's general capabilities (e.g., summarization, general classification, simple Q\&A).
  * **Weaknesses:** It can struggle with novel, highly specific, or nuanced tasks where the desired output format or logic is not obvious.
  * **Banking Use Cases:** Initial customer feedback analysis, categorizing support tickets into broad topics, drafting simple emails, identifying the general intent of a customer query.

### **Use Case 1.1: Sentiment Analysis of Bank Feedback**

**Scenario:** A bank's automated system collects feedback from a customer after a service call. We need to classify the sentiment.

#### **Naive Prompt 👎**

This prompt is too simple and gives the model no guidance on the output format or classification scheme.

In [3]:
customer_feedback_1 = "The mobile app is really slow to load my account balance, and it crashed twice yesterday. I'm very frustrated."

naive_prompt_1 = "{feedback}"

execute_prompt(
    naive_prompt_1,
    "1.1 Naive Sentiment Analysis",
    feedback=customer_feedback_1
)

--- 1.1 Naive Sentiment Analysis ---
PROMPT:
Human: The mobile app is really slow to load my account balance, and it crashed twice yesterday. I'm very frustrated.

==================== RESPONSE ====================



I'm sorry to hear that you're experiencing slow loading times and crashes with the app. Let's try a few steps to improve the situation:

1. Make sure your app is updated to the latest version.
2. Try restarting your device to clear any temporary issues.
3. Check your internet connection to ensure it's stable.
4. If the problem persists, you can try reinstalling the app.

If none of these help, please let me know your device type and app version, and I'll assist you further. Your experience is important to us, and we're here to help!


--------------------------------------------------



**Expected Response:**
The response will be a verbose rephrasing of the feedback, not a clean classification.

#### **Improved Zero-Shot Prompt 👍**

This prompt gives a clear **role**, **task**, and **strict output constraints**, guiding the model to perform a classification task accurately.

In [4]:
customer_feedback_1 = "The mobile app is really slow to load my account balance, and it crashed twice yesterday. I'm very frustrated."

zeroshot_prompt_1 = """
You are a sentiment analysis AI for a major bank. Your task is to classify customer feedback into one of three categories: Positive, Negative, or Neutral.

Analyze the following feedback and provide only the category name as the output.

Feedback: "{feedback}"
"""

execute_prompt(
    zeroshot_prompt_1,
    "1.1 Improved Zero-Shot Sentiment Analysis",
    feedback=customer_feedback_1
)

--- 1.1 Improved Zero-Shot Sentiment Analysis ---
PROMPT:
Human: 
You are a sentiment analysis AI for a major bank. Your task is to classify customer feedback into one of three categories: Positive, Negative, or Neutral.

Analyze the following feedback and provide only the category name as the output.

Feedback: "The mobile app is really slow to load my account balance, and it crashed twice yesterday. I'm very frustrated."


==================== RESPONSE ====================



Negative


--------------------------------------------------



-----

### **Use Case 1.2: Identifying Customer Intent**

**Scenario:** A chatbot needs to understand why a customer is messaging to route them to the correct department.

#### **Naive Prompt 👎**

This prompt is a vague question that will lead to a conversational and unstructured answer.

In [5]:
customer_query_1 = "Hi, I can't seem to find my last month's credit card statement in the app."

naive_prompt_2 = "What does this customer want to do? Query: {query}"

execute_prompt(
    naive_prompt_2,
    "1.2 Naive Intent Recognition",
    query=customer_query_1
)

--- 1.2 Naive Intent Recognition ---
PROMPT:
Human: What does this customer want to do? Query: Hi, I can't seem to find my last month's credit card statement in the app.

==================== RESPONSE ====================



The customer wants to access or view their last month's credit card statement in the app.


--------------------------------------------------



#### **Improved Zero-Shot Prompt 👍**

This prompt defines a set of valid intents, forcing the model to classify the query into a predefined category suitable for automated routing.

In [6]:
customer_query_1 = "Hi, I can't seem to find my last month's credit card statement in the app."

zeroshot_prompt_2 = """
You are an intent recognition model for a banking chatbot. Classify the user's query into one of the following predefined categories:
- Card Services
- Account Inquiry
- Loan Application
- Technical Support
- General Question

User Query: "{query}"

Return only the category name.
"""

execute_prompt(
    zeroshot_prompt_2,
    "1.2 Improved Zero-Shot Intent Recognition",
    query=customer_query_1
)

--- 1.2 Improved Zero-Shot Intent Recognition ---
PROMPT:
Human: 
You are an intent recognition model for a banking chatbot. Classify the user's query into one of the following predefined categories:
- Card Services
- Account Inquiry
- Loan Application
- Technical Support
- General Question

User Query: "Hi, I can't seem to find my last month's credit card statement in the app."

Return only the category name.


==================== RESPONSE ====================



Card Services


--------------------------------------------------



-----

## **2. Few-Shot Prompting: Learning from Examples**

### **Background**

**Few-shot prompting** involves providing the LLM with a few examples ("shots") of the task you want it to perform directly within the prompt. This helps the model understand the desired pattern, format, and nuances.

  * **How it works:** You show the model input-output pairs. For instance, `Input: Sea otter -> Output: Enhydra lutris. Input: Manatee -> Output: Trichechus. Input: Blue whale -> Output:`. The model learns the pattern (common name to scientific name) and completes it.
  * **Strengths:** Extremely effective for tasks requiring a specific output structure (like JSON), a particular style, or classification into custom, non-obvious categories.
  * **Weaknesses:** Requires crafting high-quality examples, and makes the prompt longer and more expensive to process.
  * **Banking Use Cases:** Extracting specific data from unstructured documents (KYC, invoices), categorizing transactions into bank-specific buckets, formatting data for API calls, drafting communications that match a brand's voice.

### **Use Case 2.1: Custom Transaction Categorization**

**Scenario:** A bank's personal finance management tool needs to categorize transactions based on its own unique category list.

#### **Naive Prompt 👎**

The model will use its own general knowledge of categories, which won't match the bank's internal system.

In [7]:
transaction_description_1 = "ZOMATO-FOOD-DELIVERY GURGAON"

naive_prompt_3 = "Categorize this transaction: '{transaction}'"

execute_prompt(
    naive_prompt_3,
    "2.1 Naive Transaction Categorization",
    transaction=transaction_description_1
)

--- 2.1 Naive Transaction Categorization ---
PROMPT:
Human: Categorize this transaction: 'ZOMATO-FOOD-DELIVERY GURGAON'

==================== RESPONSE ====================



This transaction can be categorized as **Food & Dining** or **Food Delivery**.


--------------------------------------------------



This is a good guess, but it may not match our internal category `Food & Entertainment`.

#### **Few-Shot Prompt 👍**

By providing examples, we teach the model our exact categorization schema.

In [8]:
transaction_description_1 = "ZOMATO-FOOD-DELIVERY GURGAON"

fewshot_prompt_1 = """
You are a transaction categorization engine for a bank. Your goal is to classify transactions into one of the following categories: 'Utilities', 'Groceries', 'Travel', 'Food & Entertainment', 'Shopping', 'Miscellaneous'.

Here are some examples:
1. Transaction: "AMAZON-PAY-ELECTRICITY"
   Category: Utilities
2. Transaction: "UBER-TRIP-MUMBAI-AIRPORT"
   Category: Travel
3. Transaction: "BIGBASKET-ONLINE-GROC"
   Category: Groceries

Now, categorize the following transaction. Provide only the category name.

Transaction: "{transaction}"
Category:
"""

execute_prompt(
    fewshot_prompt_1,
    "2.1 Few-Shot Transaction Categorization",
    transaction=transaction_description_1
)

--- 2.1 Few-Shot Transaction Categorization ---
PROMPT:
Human: 
You are a transaction categorization engine for a bank. Your goal is to classify transactions into one of the following categories: 'Utilities', 'Groceries', 'Travel', 'Food & Entertainment', 'Shopping', 'Miscellaneous'.

Here are some examples:
1. Transaction: "AMAZON-PAY-ELECTRICITY"
   Category: Utilities
2. Transaction: "UBER-TRIP-MUMBAI-AIRPORT"
   Category: Travel
3. Transaction: "BIGBASKET-ONLINE-GROC"
   Category: Groceries

Now, categorize the following transaction. Provide only the category name.

Transaction: "ZOMATO-FOOD-DELIVERY GURGAON"
Category:


==================== RESPONSE ====================



Food & Entertainment


--------------------------------------------------



### **Use Case 2.2: Extracting Structured Data from Text**

**Scenario:** A wealth manager receives an email from a potential client. We need to extract key details into a structured JSON object for the CRM system.

#### **Naive Prompt 👎**

This will produce an unpredictable summary, not a clean, machine-readable object.

In [9]:
client_email_1 = "Hi, my name is Rohan Gupta. I'm 42 years old and I'm interested in your wealth management services. My investment portfolio is currently around $500,000 and my risk tolerance is moderately aggressive. You can reach me at rohan.gupta@email.com."

naive_prompt_4 = "Extract the client details from this email: {email}"

execute_prompt(
    naive_prompt_4,
    "2.2 Naive Data Extraction",
    email=client_email_1
)

--- 2.2 Naive Data Extraction ---
PROMPT:
Human: Extract the client details from this email: Hi, my name is Rohan Gupta. I'm 42 years old and I'm interested in your wealth management services. My investment portfolio is currently around $500,000 and my risk tolerance is moderately aggressive. You can reach me at rohan.gupta@email.com.

==================== RESPONSE ====================



Client details extracted from the email:

- Name: Rohan Gupta  
- Age: 42 years old  
- Interest: Wealth management services  
- Investment portfolio: Approximately $500,000  
- Risk tolerance: Moderately aggressive  
- Contact email: rohan.gupta@email.com


--------------------------------------------------



#### **Few-Shot Prompt 👍**

The example teaches the model the exact JSON schema we require.

In [10]:
client_email_1 = "Hi, my name is Rohan Gupta. I'm 42 years old and I'm interested in your wealth management services. My investment portfolio is currently around $500,000 and my risk tolerance is moderately aggressive. You can reach me at rohan.gupta@email.com."

fewshot_prompt_2 = """
You are a data extraction AI that converts unstructured client emails into structured JSON objects.

---
Example 1:
Email: "My name is Priya Singh, my risk appetite is conservative and my portfolio is $1.2M. I am 55."
JSON:
{{
  "name": "Priya Singh",
  "age": 55,
  "portfolio_value": 1200000,
  "risk_profile": "Conservative",
  "contact_email": "N/A"
}}
---

Now, process the following email and provide only the JSON object as output.

Email: "{email}"
JSON:
"""

execute_prompt(
    fewshot_prompt_2,
    "2.2 Few-Shot Data Extraction",
    email=client_email_1
)

--- 2.2 Few-Shot Data Extraction ---
PROMPT:
Human: 
You are a data extraction AI that converts unstructured client emails into structured JSON objects.

---
Example 1:
Email: "My name is Priya Singh, my risk appetite is conservative and my portfolio is $1.2M. I am 55."
JSON:
{
  "name": "Priya Singh",
  "age": 55,
  "portfolio_value": 1200000,
  "risk_profile": "Conservative",
  "contact_email": "N/A"
}
---

Now, process the following email and provide only the JSON object as output.

Email: "Hi, my name is Rohan Gupta. I'm 42 years old and I'm interested in your wealth management services. My investment portfolio is currently around $500,000 and my risk tolerance is moderately aggressive. You can reach me at rohan.gupta@email.com."
JSON:


==================== RESPONSE ====================



{
  "name": "Rohan Gupta",
  "age": 42,
  "portfolio_value": 500000,
  "risk_profile": "Moderately Aggressive",
  "contact_email": "rohan.gupta@email.com"
}


--------------------------------------------------



-----

## **3. Chain-of-Thought (CoT) Prompting: Showing Your Work**

### **Background**

**Chain-of-Thought (CoT) prompting** is a technique that encourages the model to break down a complex problem into intermediate steps before giving a final answer. By simply adding phrases like "Let's think step by step," you can significantly improve performance on tasks that require logic, reasoning, or calculation.

  * **How it works:** Instead of jumping to a conclusion, the model first "thinks" out loud, detailing its reasoning process. This mimics how humans solve complex problems and often leads the model to a more accurate result.
  * **Strengths:** Massively improves accuracy on arithmetic, commonsense, and symbolic reasoning tasks. The output is more transparent and auditable, as you can see *how* the model reached its conclusion.
  * **Banking Use Cases:** Assessing loan eligibility based on multiple criteria, preliminary fraud analysis, calculating financial ratios, explaining complex financial products to customers, and checking compliance with internal policies.

### **Use Case 3.1: Preliminary Fraud Alert Analysis**

**Scenario:** A transaction is flagged for potential fraud. A junior analyst needs a summary explaining why it's suspicious.

#### **Naive Prompt 👎**

This prompt asks for a simple "yes/no" answer, which is not helpful and lacks any justification.

In [11]:
transaction_data_1 = """
- Customer: Anjali Rao
- Usual Location: Bangalore, IN
- Transaction Amount: ₹85,000
- Transaction Location: Prague, CZ
- Time: 3:15 AM IST
- Merchant Category: Luxury Watches
- Customer History: No prior international transactions. Average transaction value is ₹2,500.
"""

naive_prompt_5 = "Is this transaction likely fraudulent? Transaction Details: {transaction}"

execute_prompt(
    naive_prompt_5,
    "3.1 Naive Fraud Analysis",
    transaction=transaction_data_1
)

--- 3.1 Naive Fraud Analysis ---
PROMPT:
Human: Is this transaction likely fraudulent? Transaction Details: 
- Customer: Anjali Rao
- Usual Location: Bangalore, IN
- Transaction Amount: ₹85,000
- Transaction Location: Prague, CZ
- Time: 3:15 AM IST
- Merchant Category: Luxury Watches
- Customer History: No prior international transactions. Average transaction value is ₹2,500.


==================== RESPONSE ====================



Based on the details provided, this transaction appears suspicious and could potentially be fraudulent. Here are the key points that raise red flags:

1. **Unusual Location:** The customer, Anjali Rao, is usually located in Bangalore, India, but this transaction is taking place in Prague, Czech Republic. There is no prior history of international transactions.

2. **High Transaction Amount:** The transaction amount is ₹85,000, which is significantly higher than the customer's average transaction value of ₹2,500.

3. **Time of Transaction:** The transaction occurred at 3:15 AM IST, which is an unusual time for making purchases, especially for a luxury item.

4. **Merchant Category:** The purchase is from a luxury watches merchant category, which is a high-value and often targeted category for fraud.

Given these factors, it would be prudent to flag this transaction for further verification with the customer before approval.


--------------------------------------------------



#### **Chain-of-Thought Prompt 👍**

We explicitly ask the model to reason through the evidence step-by-step, providing a clear, auditable analysis.

In [12]:
transaction_data_1 = """
- Customer: Anjali Rao
- Usual Location: Bangalore, IN
- Transaction Amount: ₹85,000
- Transaction Location: Prague, CZ
- Time: 3:15 AM IST
- Merchant Category: Luxury Watches
- Customer History: No prior international transactions. Average transaction value is ₹2,500.
"""

cot_prompt_1 = """
You are a fraud detection analyst.
Analyze the following transaction details to determine if it is likely fraudulent.
Let's think step by step to build your analysis before making a final conclusion.

Transaction Details:
{transaction}

First, identify all the risk factors. For each factor, explain why it is suspicious.
Second, summarize your findings.
Finally, conclude with a risk assessment: LOW, MEDIUM, or HIGH.
"""

execute_prompt(
    cot_prompt_1,
    "3.1 Chain-of-Thought Fraud Analysis",
    transaction=transaction_data_1
)

--- 3.1 Chain-of-Thought Fraud Analysis ---
PROMPT:
Human: 
You are a fraud detection analyst.
Analyze the following transaction details to determine if it is likely fraudulent.
Let's think step by step to build your analysis before making a final conclusion.

Transaction Details:

- Customer: Anjali Rao
- Usual Location: Bangalore, IN
- Transaction Amount: ₹85,000
- Transaction Location: Prague, CZ
- Time: 3:15 AM IST
- Merchant Category: Luxury Watches
- Customer History: No prior international transactions. Average transaction value is ₹2,500.


First, identify all the risk factors. For each factor, explain why it is suspicious.
Second, summarize your findings.
Finally, conclude with a risk assessment: LOW, MEDIUM, or HIGH.


==================== RESPONSE ====================



Let's analyze the transaction step by step.

### Step 1: Identify all the risk factors and explain why each is suspicious.

1. **Transaction Location: Prague, CZ**
   - The customer, Anjali Rao, usually transacts in Bangalore, India.
   - This is the first international transaction recorded.
   - A sudden international transaction, especially in a distant country, is a common red flag for fraud.

2. **Transaction Amount: ₹85,000**
   - The average transaction value for the customer is ₹2,500.
   - This transaction amount is more than 30 times the average, indicating an unusual spending pattern.
   - Large, atypical transactions often indicate potential fraud.

3. **Transaction Time: 3:15 AM IST**
   - The transaction occurred at an unusual hour (early morning).
   - Fraudulent transactions often happen at odd hours when the customer is less likely to be actively monitoring their account.

4. **Merchant Category: Luxury Watches**
   - Luxury goods are high-value items that are often targeted by fraudsters.
   - This category is consistent with the high transaction amount but inconsistent with the customer's usual spending behavior.

5. **Customer History: No prior international transactions**
   - Lack of prior international transactions means this is an outlier in the customer's behavior.
   - Fraudsters often make their first fraudulent transaction abroad to avoid detection.

### Step 2: Summarize findings

- The transaction is in a foreign country where the customer has no prior history.
- The transaction amount is significantly higher than the customer's average.
- The transaction occurred at an unusual hour.
- The merchant category is a high-value luxury item, which is not consistent with the customer's typical spending.
- All these factors combined suggest a deviation from normal behavior and raise suspicion.

### Step 3: Risk assessment

Given the multiple red flags (unusual location, high amount, odd timing, and category mismatch), this transaction carries a **HIGH** risk of being fraudulent.

---

**Final conclusion:** The transaction is very likely fraudulent and should be flagged for further investigation or immediate action such as contacting the customer or temporarily blocking the transaction.


--------------------------------------------------




### **3.2. Chain-of-Thought (CoT): Evaluating a Small Business Loan**

**Scenario:** A loan officer needs a quick, reasoned assessment of a small business loan application to decide if it warrants a full underwriting process.

In [13]:
loan_application_data = """
- Business Name: 'Global Exports Inc.'
- Years in Business: 2
- Annual Revenue: ₹75,00,000
- Requested Loan Amount: ₹20,00,000
- Owner's CIBIL Score: 680
---
Bank's Minimum Criteria:
- Minimum Years in Business: 3
- Minimum Annual Revenue: ₹50,00,000
- Minimum CIBIL Score: 720
"""

#### **Naive Prompt 👎**

A direct question will yield a "black box" answer with minimal justification, making it hard to trust or audit.

In [14]:
naive_prompt_3 = "Based on the data and criteria, should we approve a loan for Global Exports Inc.? \n\n{data}"

execute_prompt(
    naive_prompt_3,
    "3.1 Naive Loan Assessment",
    data=loan_application_data
)

--- 3.1 Naive Loan Assessment ---
PROMPT:
Human: Based on the data and criteria, should we approve a loan for Global Exports Inc.? 


- Business Name: 'Global Exports Inc.'
- Years in Business: 2
- Annual Revenue: ₹75,00,000
- Requested Loan Amount: ₹20,00,000
- Owner's CIBIL Score: 680
---
Bank's Minimum Criteria:
- Minimum Years in Business: 3
- Minimum Annual Revenue: ₹50,00,000
- Minimum CIBIL Score: 720


==================== RESPONSE ====================



Based on the provided data and the bank's minimum criteria:

- **Years in Business:** Global Exports Inc. has been in business for 2 years, but the minimum required is 3 years. **Does not meet criteria.**
- **Annual Revenue:** ₹75,00,000, which is above the minimum required ₹50,00,000. **Meets criteria.**
- **Owner's CIBIL Score:** 680, which is below the minimum required 720. **Does not meet criteria.**

Since Global Exports Inc. does not meet two of the three key criteria (Years in Business and CIBIL Score), **the loan should not be approved** based on the bank's minimum requirements.


--------------------------------------------------




*Critique:* The answer is correct, but it lacks a structured breakdown of the decision process.

#### **Improved CoT Prompt 👍**

This prompt forces the model to perform a step-by-step analysis, comparing each application data point against the criteria before reaching a conclusion.

In [15]:
cot_prompt_3 = """
You are a junior loan underwriter AI. Evaluate the small business loan application against the bank's criteria. Let's think step by step to create a clear evaluation report.

Application Data and Criteria:
{data}

**Evaluation Process:**
1.  **Check Years in Business:** Compare the applicant's years in business against the minimum requirement. State Pass or Fail.
2.  **Check Annual Revenue:** Compare the applicant's revenue against the minimum requirement. State Pass or Fail.
3.  **Check CIBIL Score:** Compare the owner's CIBIL score against the minimum requirement. State Pass or Fail.
4.  **Summary of Findings:** Briefly list the strengths and weaknesses of the application.
5.  **Final Recommendation:** Conclude with 'Proceed to Underwriting' or 'Reject'.
"""

execute_prompt(
    cot_prompt_3,
    "3.2 CoT Loan Assessment",
    data=loan_application_data
)

--- 3.2 CoT Loan Assessment ---
PROMPT:
Human: 
You are a junior loan underwriter AI. Evaluate the small business loan application against the bank's criteria. Let's think step by step to create a clear evaluation report.

Application Data and Criteria:

- Business Name: 'Global Exports Inc.'
- Years in Business: 2
- Annual Revenue: ₹75,00,000
- Requested Loan Amount: ₹20,00,000
- Owner's CIBIL Score: 680
---
Bank's Minimum Criteria:
- Minimum Years in Business: 3
- Minimum Annual Revenue: ₹50,00,000
- Minimum CIBIL Score: 720


**Evaluation Process:**
1.  **Check Years in Business:** Compare the applicant's years in business against the minimum requirement. State Pass or Fail.
2.  **Check Annual Revenue:** Compare the applicant's revenue against the minimum requirement. State Pass or Fail.
3.  **Check CIBIL Score:** Compare the owner's CIBIL score against the minimum requirement. State Pass or Fail.
4.  **Summary of Findings:** Briefly list the strengths and weaknesses of the applicat

**Evaluation Report for Global Exports Inc.**

1. **Check Years in Business:**  
   - Applicant's Years in Business: 2  
   - Minimum Required: 3  
   - **Result:** Fail (Applicant does not meet the minimum years in business requirement)

2. **Check Annual Revenue:**  
   - Applicant's Annual Revenue: ₹75,00,000  
   - Minimum Required: ₹50,00,000  
   - **Result:** Pass (Applicant meets the minimum annual revenue requirement)

3. **Check CIBIL Score:**  
   - Applicant's CIBIL Score: 680  
   - Minimum Required: 720  
   - **Result:** Fail (Applicant does not meet the minimum CIBIL score requirement)

4. **Summary of Findings:**  
   - **Strengths:** The business has a strong annual revenue of ₹75,00,000, which exceeds the minimum requirement.  
   - **Weaknesses:** The business has only been operating for 2 years, which is below the minimum of 3 years. Additionally, the owner's CIBIL score of 680 is below the required 720, indicating a potential credit risk.

5. **Final Recommendation:**  
   - **Reject** due to failure to meet the minimum criteria for years in business and CIBIL score.


--------------------------------------------------




*Critique:* The response is transparent, auditable, and provides a clear, well-supported rationale for the decision.


-----

## **4. Tree-of-Thought (ToT) Prompting: Exploring Multiple Paths**

### **Background**

**Tree-of-Thought (ToT)** is an advanced technique that generalizes Chain-of-Thought. Instead of just one linear reasoning path, ToT encourages the model to explore *multiple* different lines of reasoning (branches of a tree). It can then self-evaluate these paths and choose the most promising one to pursue.

  * **How it works:** A full ToT implementation is complex and often requires an agentic framework with multiple LLM calls. However, we can simulate the *spirit* of ToT in a single prompt by asking the model to:
    1.  **Generate** multiple potential solutions or viewpoints.
    2.  **Evaluate** the pros and cons of each.
    3.  **Select** the best option based on that evaluation.
  * **Strengths:** Exceptionally powerful for complex problems with no single right answer, such as strategic planning, creative problem-solving, and generating robust recommendations.
  * **Banking Use Cases:** Developing personalized financial plans for clients, creating customer retention strategies, brainstorming new product features, and formulating responses to complex regulatory inquiries.

### **Use Case 4.1: Proposing a Client Investment Strategy**

**Scenario:** A financial advisor needs to propose an investment strategy for a young client with a moderate risk appetite.

#### **Naive Prompt 👎**

This gives a generic, one-size-fits-all answer without considering alternatives or trade-offs.

In [16]:
client_profile_1 = """
- Name: Sameer Khan
- Age: 28
- Goal: Wealth accumulation for early retirement
- Risk Tolerance: Moderate
- Annual Income: ₹25,00,000
"""

naive_prompt_6 = "Suggest an investment portfolio for this client: {profile}"

execute_prompt(
    naive_prompt_6,
    "4.1 Naive Investment Suggestion",
    profile=client_profile_1
)

--- 4.1 Naive Investment Suggestion ---
PROMPT:
Human: Suggest an investment portfolio for this client: 
- Name: Sameer Khan
- Age: 28
- Goal: Wealth accumulation for early retirement
- Risk Tolerance: Moderate
- Annual Income: ₹25,00,000


==================== RESPONSE ====================



Certainly! For Sameer Khan, age 28, aiming for wealth accumulation for early retirement with a moderate risk tolerance and a healthy annual income of ₹25,00,000, the investment portfolio should balance growth potential with some stability. Here’s a suggested diversified portfolio:

---

### Investment Portfolio Suggestion for Sameer Khan

**1. Equity Mutual Funds / Stocks (60%)**  
- **Why:** Equities offer higher growth potential over the long term, suitable for wealth accumulation at a young age.  
- **Allocation:** ₹15,00,000 annually  
- **Breakdown:**  
  - Large-cap funds/stocks (30%) – Stable companies with steady growth  
  - Mid-cap funds/stocks (20%) – Higher growth potential with moderate risk  
  - Multi-cap or flexi-cap funds (10%) – Diversified exposure across market caps  

**2. Debt Instruments (20%)**  
- **Why:** To reduce portfolio volatility and provide stability.  
- **Allocation:** ₹5,00,000 annually  
- **Options:**  
  - Corporate bonds or bond funds  
  - Public Provident Fund (PPF) – tax-efficient and safe  
  - Fixed Deposits (FDs) with good interest rates  

**3. Hybrid Funds / Balanced Advantage Funds (10%)**  
- **Why:** These funds dynamically manage equity and debt allocation, balancing risk and return.  
- **Allocation:** ₹2,50,000 annually  

**4. Alternative Investments / Gold (5%)**  
- **Why:** Diversification and hedge against inflation and market volatility.  
- **Allocation:** ₹1,25,000 annually  
- **Options:**  
  - Sovereign Gold Bonds (SGBs)  
  - Gold ETFs  

**5. Emergency Fund & Liquid Assets (5%)**  
- **Why:** To maintain liquidity for unforeseen expenses without disturbing long-term investments.  
- **Allocation:** ₹1,25,000 annually  
- **Options:**  
  - Liquid mutual funds  
  - Savings account with high interest  

---

### Additional Recommendations:

- **Systematic Investment Plan (SIP):** Invest monthly through SIPs to benefit from rupee cost averaging and disciplined investing.  
- **Tax Planning:** Utilize tax-saving instruments like ELSS funds, PPF, and NPS to optimize tax outgo under Section 80C.  
- **Review Portfolio Annually:** Rebalance the portfolio yearly to maintain the desired asset allocation and adjust for changing goals or market conditions.  
- **Retirement Corpus Estimation:** Assuming Sameer wants to retire early (say at 45), with a moderate risk portfolio averaging 10-12% returns, consistent investing of ₹25 lakh annually (or a portion of it) can help accumulate a substantial corpus.

---

If you want, I can also help suggest specific funds or stocks based on current market conditions and Sameer’s preferences. Would you like that?


--------------------------------------------------



#### **"ToT-style" Prompt 👍**

This prompt guides the model to explore and critique several strategies before making a final, well-justified recommendation.

In [17]:
client_profile_1 = """
- Name: Sameer Khan
- Age: 28
- Goal: Wealth accumulation for early retirement
- Risk Tolerance: Moderate
- Annual Income: ₹25,00,000
"""

tot_prompt_1 = """
You are a senior investment strategist. Your task is to devise an investment strategy for a new client.
Use the following structured thinking process:

Client Profile:
{profile}

1.  **Propose Three Distinct Strategies:** Based on the client's profile, generate three different investment strategies. Name each one (e.g., 'Balanced Growth', 'Aggressive Tech Focus', 'Diversified Core'). For each, suggest a sample asset allocation (Equities, Debt, Alternatives).
2.  **Evaluate Each Strategy:** For each of the three strategies, briefly list the potential Pros and Cons specifically for this client. Consider their age, goal, and risk tolerance.
3.  **Recommend and Justify:** Based on your evaluation, select the single most suitable strategy for the client and write a concluding paragraph explaining why it is the best choice.
"""

execute_prompt(
    tot_prompt_1,
    "4.1 ToT-style Investment Strategy",
    profile=client_profile_1
)

--- 4.1 ToT-style Investment Strategy ---
PROMPT:
Human: 
You are a senior investment strategist. Your task is to devise an investment strategy for a new client.
Use the following structured thinking process:

Client Profile:

- Name: Sameer Khan
- Age: 28
- Goal: Wealth accumulation for early retirement
- Risk Tolerance: Moderate
- Annual Income: ₹25,00,000


1.  **Propose Three Distinct Strategies:** Based on the client's profile, generate three different investment strategies. Name each one (e.g., 'Balanced Growth', 'Aggressive Tech Focus', 'Diversified Core'). For each, suggest a sample asset allocation (Equities, Debt, Alternatives).
2.  **Evaluate Each Strategy:** For each of the three strategies, briefly list the potential Pros and Cons specifically for this client. Consider their age, goal, and risk tolerance.
3.  **Recommend and Justify:** Based on your evaluation, select the single most suitable strategy for the client and write a concluding paragraph explaining why it is the

Certainly! Below is a structured investment strategy proposal for Sameer Khan, considering his profile and goals.

---

### Client Profile Recap:
- **Name:** Sameer Khan  
- **Age:** 28  
- **Goal:** Wealth accumulation for early retirement (likely 10-15 years horizon)  
- **Risk Tolerance:** Moderate  
- **Annual Income:** ₹25,00,000  

---

## 1. Propose Three Distinct Strategies

### Strategy 1: Balanced Growth
- **Equities:** 60% (mix of large-cap, mid-cap, and some international exposure)  
- **Debt:** 30% (government bonds, high-quality corporate bonds, fixed deposits)  
- **Alternatives:** 10% (REITs, gold ETFs, and some exposure to private equity or start-ups via platforms)  

### Strategy 2: Aggressive Tech Focus
- **Equities:** 80% (heavy focus on technology, innovation sectors, and emerging markets)  
- **Debt:** 10% (short-term debt instruments for liquidity and risk mitigation)  
- **Alternatives:** 10% (cryptocurrency, venture capital funds, and thematic ETFs)  

### Strategy 3: Diversified Core
- **Equities:** 50% (diversified across sectors and geographies, including dividend-paying stocks)  
- **Debt:** 40% (mix of fixed income, including inflation-indexed bonds and corporate bonds)  
- **Alternatives:** 10% (gold, real estate funds, and infrastructure investments)  

---

## 2. Evaluate Each Strategy

### Strategy 1: Balanced Growth
- **Pros:**  
  - Aligns well with moderate risk tolerance.  
  - Equities provide growth potential; debt cushions volatility.  
  - Alternatives add diversification and inflation hedge.  
  - Suitable for a 10-15 year horizon aiming for wealth accumulation.  
- **Cons:**  
  - May underperform highly aggressive strategies in bull markets.  
  - Moderate exposure to alternatives may not fully capitalize on high-growth opportunities.

### Strategy 2: Aggressive Tech Focus
- **Pros:**  
  - High growth potential, suitable for early retirement goal.  
  - Exposure to cutting-edge sectors could generate outsized returns.  
  - Younger age supports taking on more risk.  
- **Cons:**  
  - High volatility and risk may not suit moderate risk tolerance.  
  - Concentration risk in tech and emerging sectors.  
  - Potential for significant drawdowns, which could impact investment horizon.

### Strategy 3: Diversified Core
- **Pros:**  
  - Strong capital preservation focus with higher debt allocation.  
  - Diversification reduces risk and smooths returns.  
  - Suitable for investors who want steady, less volatile growth.  
- **Cons:**  
  - Lower equity allocation may limit wealth accumulation potential.  
  - May not fully meet early retirement wealth targets within the desired timeline.  
  - Alternatives allocation is conservative, limiting upside.

---

## 3. Recommend and Justify

**Recommended Strategy: Balanced Growth**

Given Sameer’s age (28), goal of early retirement, and moderate risk tolerance, the **Balanced Growth** strategy strikes the optimal balance between growth and risk management. It provides sufficient equity exposure to capitalize on long-term market appreciation, while the debt component offers stability and downside protection. The inclusion of alternatives like REITs and gold ETFs adds diversification and inflation hedging, which is crucial for preserving purchasing power over a 10-15 year horizon. This approach aligns well with Sameer’s moderate risk appetite and wealth accumulation goal, offering a disciplined yet growth-oriented path to early retirement without exposing him to the high volatility and concentration risks inherent in more aggressive strategies.

---

If you would like, I can also provide specific fund or instrument recommendations within these allocations or a sample portfolio.


--------------------------------------------------




-----

### **4.2. Tree-of-Thought (ToT): Devising a Debt Restructuring Plan**

**Scenario:** A financial advisor is helping a client who is overwhelmed with multiple sources of debt. The goal is to propose and compare different concrete strategies for tackling the debt.

In [18]:
client_debt_profile = """
- Client: Priya
- Monthly Take-home Salary: ₹80,000
- Monthly Expenses (non-debt): ₹45,000
- Available for Debt Repayment: ₹35,000 per month
- Debts:
  1. Credit Card: ₹2,00,000 at 36% APR
  2. Personal Loan: ₹5,00,000 at 14% APR
  3. Car Loan: ₹3,00,000 at 9% APR
"""

#### **Naive Prompt 👎**

This prompt will generate generic, non-comparative advice that isn't tailored into actionable, alternative strategies.

In [19]:
naive_prompt_4 = "How can this client manage their debt? \n\n{profile}"

execute_prompt(
    naive_prompt_4,
    "4.1 Naive Debt Advice",
    profile=client_debt_profile
)

--- 4.1 Naive Debt Advice ---
PROMPT:
Human: How can this client manage their debt? 


- Client: Priya
- Monthly Take-home Salary: ₹80,000
- Monthly Expenses (non-debt): ₹45,000
- Available for Debt Repayment: ₹35,000 per month
- Debts:
  1. Credit Card: ₹2,00,000 at 36% APR
  2. Personal Loan: ₹5,00,000 at 14% APR
  3. Car Loan: ₹3,00,000 at 9% APR


==================== RESPONSE ====================



To help Priya manage her debt effectively, the goal is to minimize interest payments and pay off high-interest debt quickly while maintaining financial stability. Here's a step-by-step approach based on her financial situation:

---

### Summary of Priya's Financials:
- **Monthly Take-home Salary:** ₹80,000
- **Monthly Expenses (non-debt):** ₹45,000
- **Available for Debt Repayment:** ₹35,000
- **Debts:**
  1. Credit Card: ₹2,00,000 at 36% APR (very high interest)
  2. Personal Loan: ₹5,00,000 at 14% APR
  3. Car Loan: ₹3,00,000 at 9% APR

---

### Step 1: Prioritize Debt Repayment by Interest Rate (Highest First)
- Credit Card (36%) → Personal Loan (14%) → Car Loan (9%)

High-interest debt grows fastest, so paying off the credit card first will reduce interest burden significantly.

---

### Step 2: Allocate Monthly Debt Repayment
- Total available for debt repayment: ₹35,000

**Suggested allocation:**

- Pay **minimum required payments** on Personal Loan and Car Loan to avoid penalties (assumed minimums, but if not known, estimate roughly 2-3% of principal):
  - Personal Loan minimum: ~₹10,000 (estimate)
  - Car Loan minimum: ~₹5,000 (estimate)
  
- Remaining for Credit Card: ₹35,000 - ₹10,000 - ₹5,000 = ₹20,000

If minimum payments are lower, allocate more to credit card.

---

### Step 3: Accelerate Credit Card Repayment
- Pay ₹20,000+ monthly towards credit card debt.
- At 36% APR, ₹2,00,000 will reduce faster with aggressive payments.
- Once credit card is fully paid, redirect that amount to personal loan.

---

### Step 4: After Credit Card is Paid Off
- Use the full ₹35,000 to pay down Personal Loan (₹5,00,000 at 14% APR).
- This will reduce principal faster and save interest.

---

### Step 5: Finally, Pay Off Car Loan
- After personal loan is cleared, use the full ₹35,000 to pay off car loan.

---

### Step 6: Consider Additional Strategies
- **Balance Transfer:** If possible, transfer credit card balance to a lower-interest personal loan or a 0% interest balance transfer card.
- **Debt Consolidation:** Consider consolidating personal loan and credit card debt into a single loan with a lower interest rate.
- **Increase Income / Reduce Expenses:** If possible, increase monthly debt repayment amount.
- **Emergency Fund:** Maintain some savings to avoid new debt in emergencies.

---

### Estimated Timeline (Approximate)
- Credit Card (₹2,00,000 at 36%): Paying ₹20,000/month → roughly 11-12 months.
- Personal Loan (₹5,00,000 at 14%): Paying ₹35,000/month → approx. 16-18 months after credit card.
- Car Loan (₹3,00,000 at 9%): Paying ₹35,000/month → approx. 9-10 months after personal loan.

Total debt-free timeline: ~3.5 years.

---

### Summary Table

| Debt           | Principal | Interest Rate | Monthly Payment (min est.) | Priority | Notes                      |
|----------------|-----------|---------------|----------------------------|----------|----------------------------|
| Credit Card    | ₹2,00,000 | 36%           | ₹20,000+                   | 1        | Pay aggressively first     |
| Personal Loan  | ₹5,00,000 | 14%           | ₹10,000                    | 2        | Pay minimum, then full     |
| Car Loan       | ₹3,00,000 | 9%            | ₹5,000                     | 3        | Pay minimum, then full     |

---

If Priya can stick to this plan, she will minimize interest payments and become debt-free in a reasonable timeframe.


--------------------------------------------------



*Critique:* The advice is sound but lacks structure and fails to explore or compare alternative, well-known strategies.

#### **Improved "ToT-style" Prompt 👍**

This prompt instructs the model to generate and evaluate multiple distinct strategies, presenting a comprehensive analysis of the options.

In [20]:
tot_prompt_4 = """
You are an expert financial advisor. For the client profile below, devise a debt management plan using the following structured thinking process.

Client Profile:
{profile}

1.  **Propose Three Distinct Strategies:** Generate three different debt repayment strategies. Name each one (e.g., 'Debt Avalanche', 'Debt Snowball', 'Loan Consolidation'). Briefly explain how each would work for this client.
2.  **Evaluate Each Strategy:** For each of the three strategies, list the primary 'Pro' (e.g., lowest total interest paid) and the primary 'Con' (e.g., may feel slower) for this specific client.
3.  **Recommend and Justify:** Based on your evaluation, select the single most suitable strategy and explain why it is the best recommendation for someone in this situation.
"""

execute_prompt(
    tot_prompt_4,
    "4.2 ToT-style Debt Plan",
    profile=client_debt_profile
)

--- 4.2 ToT-style Debt Plan ---
PROMPT:
Human: 
You are an expert financial advisor. For the client profile below, devise a debt management plan using the following structured thinking process.

Client Profile:

- Client: Priya
- Monthly Take-home Salary: ₹80,000
- Monthly Expenses (non-debt): ₹45,000
- Available for Debt Repayment: ₹35,000 per month
- Debts:
  1. Credit Card: ₹2,00,000 at 36% APR
  2. Personal Loan: ₹5,00,000 at 14% APR
  3. Car Loan: ₹3,00,000 at 9% APR


1.  **Propose Three Distinct Strategies:** Generate three different debt repayment strategies. Name each one (e.g., 'Debt Avalanche', 'Debt Snowball', 'Loan Consolidation'). Briefly explain how each would work for this client.
2.  **Evaluate Each Strategy:** For each of the three strategies, list the primary 'Pro' (e.g., lowest total interest paid) and the primary 'Con' (e.g., may feel slower) for this specific client.
3.  **Recommend and Justify:** Based on your evaluation, select the single most suitable strategy 

Certainly! Let’s analyze Priya’s debt situation and devise a structured debt management plan.

---

### Client Summary:
- **Monthly Take-home Salary:** ₹80,000  
- **Monthly Expenses (non-debt):** ₹45,000  
- **Available for Debt Repayment:** ₹35,000  
- **Debts:**
  1. Credit Card: ₹2,00,000 @ 36% APR  
  2. Personal Loan: ₹5,00,000 @ 14% APR  
  3. Car Loan: ₹3,00,000 @ 9% APR  

---

## 1. Propose Three Distinct Strategies

### Strategy A: Debt Avalanche  
- **How it works:** Prioritize paying off the debt with the highest interest rate first while making minimum payments on others.  
- For Priya: Pay minimum on Personal and Car loans, put maximum ₹35,000 towards Credit Card debt first (36% APR). Once Credit Card is cleared, move to Personal Loan (14%), then Car Loan (9%).

### Strategy B: Debt Snowball  
- **How it works:** Prioritize paying off the smallest debt first to build momentum and motivation, regardless of interest rate.  
- For Priya: Pay off Credit Card first (₹2,00,000), then Car Loan (₹3,00,000), then Personal Loan (₹5,00,000), putting ₹35,000 towards the smallest debt and minimum on others.

### Strategy C: Loan Consolidation (Refinancing)  
- **How it works:** Combine high-interest debts into a single loan with a lower interest rate, reducing monthly interest burden and simplifying payments.  
- For Priya: Explore options to consolidate Credit Card and Personal Loan into a lower-interest personal loan (say ~12% APR), then pay Car Loan separately. Use ₹35,000 to pay consolidated loan aggressively.

---

## 2. Evaluate Each Strategy

| Strategy          | Primary Pro                                               | Primary Con                                               |
|-------------------|-----------------------------------------------------------|-----------------------------------------------------------|
| **Debt Avalanche** | Minimizes total interest paid over time (especially critical with 36% credit card debt). | May feel slow initially as Credit Card debt is large; no quick wins for motivation. |
| **Debt Snowball**  | Provides quick psychological wins by clearing smaller debts first (Credit Card here is smallest). | Likely to pay more interest overall, especially on high-interest credit card debt. |
| **Loan Consolidation** | Simplifies payments and lowers interest rates, potentially reducing total interest and monthly burden. | May require good credit score; consolidation loan interest might still be high; fees and processing time involved. |

---

## 3. Recommend and Justify

**Recommended Strategy: Debt Avalanche**

**Justification:**

- Priya’s highest interest debt is the Credit Card at 36% APR, which is extremely expensive and will accumulate interest rapidly.
- By focusing the ₹35,000 monthly repayment on the Credit Card first, she will reduce the principal quickly, thereby minimizing the total interest paid.
- Although the Credit Card debt is not the smallest, the financial benefit of reducing this high-interest debt first outweighs the psychological benefit of quick wins.
- Once the Credit Card is paid off, she can redirect the entire ₹35,000 to the Personal Loan (14%), then finally the Car Loan (9%), accelerating payoff.
- Loan consolidation might be beneficial but depends on eligibility and available offers; it’s worth exploring but should not delay aggressive repayment of the high-interest credit card.
- Debt Snowball is less optimal here because the Credit Card debt is already the smallest, so it aligns with Avalanche in order but Avalanche focuses on interest rates explicitly.

---

### Additional Recommendations:

- **Avoid new credit card usage** until the balance is cleared to prevent interest accumulation.
- **Build an emergency fund** once debts are under control to avoid future reliance on high-interest credit.
- **Review consolidation options** with banks or NBFCs to see if a lower interest rate loan is available.
- **Track monthly progress** and adjust repayment amounts if expenses or income change.

---

If you want, I can also provide an estimated timeline and interest cost for the Debt Avalanche approach. Would you like that?


--------------------------------------------------




*Critique:* This response provides a strategic, multi-faceted analysis that empowers the client and advisor to make an informed decision based on clear trade-offs.

---

## Part 2 — Chain-of-Verification (CoVe)

We've seen Zero-Shot, Few-Shot, CoT, and ToT-style prompting. Now we add a **verification step** — the model double-checks its own draft against the source before producing the final answer. Empirically, CoVe substantially reduces hallucinations on document summarization and factual analysis.

**The 4-step CoVe pattern:**
1. **Draft** — generate an initial response.
2. **Plan verification questions** — what facts does the draft claim?
3. **Answer each question independently** — using only the source.
4. **Revise** — produce a final response consistent with the verified answers.


-----

## **Chain-of-Verification (CoVe): Minimizing Hallucinations**

### **Background**

Factual accuracy is non-negotiable in banking. **Chain-of-Verification (CoVe)** is a technique designed to make an LLM double-check its own work, significantly reducing the risk of generating plausible-sounding but incorrect information (hallucinations).

  * **How it works:** It's a deliberate, multi-step process.
    1.  **Draft Response:** The LLM generates an initial answer to the query.
    2.  **Plan Verifications:** The model reviews its own draft and generates a list of factual claims that need to be checked.
    3.  **Execute Verifications:** The model answers these verification questions *independently*, which prevents the initial draft from biasing the check.
    4.  **Final Verified Response:** The model revises the initial draft based on the outcome of the verifications, correcting any inaccuracies.
  * **Strengths:** Drastically improves the factual grounding and reliability of generated text. It's an essential technique for high-stakes content generation.
  * **Banking Use Cases:** Summarizing complex regulatory documents (like RBI circulars), generating financial reports, creating internal policy documentation, or drafting market analysis where every claim must be backed by data.

### **Use Case 1: Summarizing a Regulatory Guideline**

**Scenario:** A compliance officer needs a quick, accurate summary of a new (fictional) RBI guideline on digital lending.

In [21]:
rbi_guideline_text = """
RBI Circular No. 2025/11/A-34
Subject: Guidelines on Digital Lending for Non-Banking Financial Companies (NBFCs)

1. Introduction: To ensure customer protection and data security, all NBFCs engaged in digital lending must adhere to the following.
2. Loan Disbursal: All loan disbursals must be executed directly from the NBFC's bank account to the borrower's bank account. No third-party pool accounts are permitted. This rule is effective from December 1, 2025.
3. Data Privacy: Only minimal, necessary customer data may be collected. Explicit customer consent is required for any data sharing. Biometric data cannot be stored.
4. Applicability: These guidelines apply to all NBFCs. Regional Rural Banks (RRBs) are currently exempt.
"""

#### **Naive Prompt 👎**

A simple summarization request can lead to subtle but critical errors of omission or misinterpretation.

In [22]:
naive_prompt_cove_1 = "Summarize the key points of the following RBI guideline: \n\n{document}"
naive_summary = execute_prompt(
    naive_prompt_cove_1,
    "2.1 Naive Guideline Summary",
    document=rbi_guideline_text
)

--- 2.1 Naive Guideline Summary ---
PROMPT:
Human: Summarize the key points of the following RBI guideline: 


RBI Circular No. 2025/11/A-34
Subject: Guidelines on Digital Lending for Non-Banking Financial Companies (NBFCs)

1. Introduction: To ensure customer protection and data security, all NBFCs engaged in digital lending must adhere to the following.
2. Loan Disbursal: All loan disbursals must be executed directly from the NBFC's bank account to the borrower's bank account. No third-party pool accounts are permitted. This rule is effective from December 1, 2025.
3. Data Privacy: Only minimal, necessary customer data may be collected. Explicit customer consent is required for any data sharing. Biometric data cannot be stored.
4. Applicability: These guidelines apply to all NBFCs. Regional Rural Banks (RRBs) are currently exempt.


==================== RESPONSE ====================



Here are the key points of RBI Circular No. 2025/11/A-34 on Digital Lending for NBFCs:

1. **Purpose:** To ensure customer protection and data security in digital lending by NBFCs.
2. **Loan Disbursal:** Loans must be disbursed directly from the NBFC’s bank account to the borrower’s bank account. Use of third-party pool accounts is prohibited. This rule takes effect from December 1, 2025.
3. **Data Privacy:** Only essential customer data should be collected, with explicit consent required for any data sharing. Storage of biometric data is not allowed.
4. **Scope:** The guidelines apply to all NBFCs, while Regional Rural Banks (RRBs) are exempt for now.


--------------------------------------------------




**Critique:** The summary is mostly correct, but it makes one critical mistake: it claims the rules apply to "all financial institutions," when the text clearly exempts Regional Rural Banks (RRBs).

#### **CoVe Implementation 👍**

We simulate the CoVe process using a sequence of prompts to force self-correction.

In [ ]:
# --- CoVe Step 1: Draft Initial Response ---
step1_prompt = "Draft a summary of the key points of the following RBI guideline: \n\n{document}"
draft_summary = execute_prompt(
    step1_prompt,
    "CoVe Step 1: Draft Summary",
    document=rbi_guideline_text
)

# --- CoVe Step 2: Plan Verifications ---
step2_prompt = """
Based on the following summary, generate a list of verification questions to fact-check its claims against the original document.

Summary:
{summary}
"""
verification_plan = execute_prompt(
    step2_prompt,
    "CoVe Step 2: Plan Verifications",
    summary=draft_summary
)

# --- CoVe Step 3: Execute Verifications (Simulated) ---
# In a real system, you'd answer these questions independently. Here, we bundle them.
step3_prompt = """
Answer the following questions based *only* on the provided RBI document.

Document:
{document}

Questions:
{questions}
"""
verification_results = execute_prompt(
    step3_prompt,
    "CoVe Step 3: Execute Verifications",
    document=rbi_guideline_text,
    questions=verification_plan
)

# --- CoVe Step 4: Generate Final Verified Response ---
step4_prompt = """
You have an initial draft summary and a set of verified facts.
Your task is to rewrite the summary, correcting any inaccuracies or omissions based on the verified facts.

Initial Summary:
{summary}

Verified Facts (Q&A):
{verifications}

Final Verified Summary:
"""
final_summary = execute_prompt(
    step4_prompt,
    "CoVe Step 4: Final Verified Summary",
    summary=draft_summary,
    verifications=verification_results
)

--- CoVe Step 1: Draft Summary ---
PROMPT:
Human: Draft a summary of the key points of the following RBI guideline: 


RBI Circular No. 2025/11/A-34
Subject: Guidelines on Digital Lending for Non-Banking Financial Companies (NBFCs)

1. Introduction: To ensure customer protection and data security, all NBFCs engaged in digital lending must adhere to the following.
2. Loan Disbursal: All loan disbursals must be executed directly from the NBFC's bank account to the borrower's bank account. No third-party pool accounts are permitted. This rule is effective from December 1, 2025.
3. Data Privacy: Only minimal, necessary customer data may be collected. Explicit customer consent is required for any data sharing. Biometric data cannot be stored.
4. Applicability: These guidelines apply to all NBFCs. Regional Rural Banks (RRBs) are currently exempt.


==================== RESPONSE ====================

The Reserve Bank of India (RBI) has issued Circular No. 2025/11/A-34, outlining new guideline

This final version correctly captures the nuance of the exemption, which the naive prompt missed.

### **Chain-of-Verification (CoVe) Use Case 2: Generating Factual Stock Analysis**

**Scenario:** A junior equity analyst needs to write a short, factually precise summary of a company's quarterly performance for an internal client newsletter. The summary must not contain any speculation.

In [24]:
# Fictional quarterly data points for "Innovate Corp."
stock_data = """
- Company: Innovate Corp.
- Revenue: ₹150 Crores (an increase of 12% Year-over-Year).
- Net Profit: ₹25 Crores (an increase of 8% Year-over-Year).
- Key Highlight: Successfully launched new AI platform 'Cognito'.
- Stock Price Movement (last quarter): -5%.
- Analyst Consensus Rating: 'Hold'.
"""

#### **Naive Prompt 👎**

The model might try to be "helpful" by inventing a narrative or a cause-and-effect relationship that isn't supported by the facts provided.

In [25]:
naive_prompt_cove_2 = "Write a short paragraph analyzing Innovate Corp's latest quarterly results based on these data points: \n\n{data}"
naive_analysis = execute_prompt(
    naive_prompt_cove_2,
    "2.2 Naive Stock Analysis",
    data=stock_data
)

--- 2.2 Naive Stock Analysis ---
PROMPT:
Human: Write a short paragraph analyzing Innovate Corp's latest quarterly results based on these data points: 


- Company: Innovate Corp.
- Revenue: ₹150 Crores (an increase of 12% Year-over-Year).
- Net Profit: ₹25 Crores (an increase of 8% Year-over-Year).
- Key Highlight: Successfully launched new AI platform 'Cognito'.
- Stock Price Movement (last quarter): -5%.
- Analyst Consensus Rating: 'Hold'.


==================== RESPONSE ====================



Innovate Corp. demonstrated solid financial performance in the latest quarter, with revenue rising 12% year-over-year to ₹150 Crores and net profit increasing by 8% to ₹25 Crores. The successful launch of their new AI platform, 'Cognito', marks a significant strategic milestone that could drive future growth. However, despite these positive fundamentals, the stock price declined by 5% over the quarter, reflecting some market caution. This sentiment is echoed by analysts, who have maintained a 'Hold' rating, suggesting that while the company shows promise, investors may be awaiting clearer indications of sustained momentum before committing further.


--------------------------------------------------



**Critique:** The phrase "market reacted skeptically" directly and incorrectly links the stock drop to the results, a causal relationship not present in the source data. This is speculation, not factual reporting.

#### **CoVe Implementation 👍**

The CoVe process forces the model to check its own claims against the source data, stripping out any speculative connections.

In [26]:
# --- CoVe Step 1: Draft Initial Response ---
step1_prompt = "Draft a short paragraph analyzing Innovate Corp's latest quarterly results based on these data points: \n\n{data}"
draft_analysis = execute_prompt(
    step1_prompt,
    "CoVe Step 1: Draft Analysis",
    data=stock_data
)

# --- CoVe Step 2: Plan Verifications ---
step2_prompt = """
Based on the following draft analysis, generate a list of verification questions to fact-check its claims against the original data points.

Draft Analysis:
{analysis}
"""
verification_plan_2 = execute_prompt(
    step2_prompt,
    "CoVe Step 2: Plan Verifications",
    analysis=draft_analysis
)

# --- CoVe Step 3: Execute Verifications ---
step3_prompt = """
Answer the following questions based *only* on the provided data points. 
Do not infer or speculate.

Data Points:
{data}

Questions:
{questions}
"""
verification_results_2 = execute_prompt(
    step3_prompt,
    "CoVe Step 3: Execute Verifications",
    data=stock_data,
    questions=verification_plan_2
)

# --- CoVe Step 4: Generate Final Verified Response ---
step4_prompt = """
You have a draft analysis and a set of verified facts.
Your task is to rewrite the analysis, ensuring every statement is directly supported by the verified facts.
Remove any speculation or causal links that are not explicitly mentioned in the data.

Draft Analysis:
{analysis}

Verified Facts (Q&A):
{verifications}

Final Verified Analysis:
"""
final_analysis = execute_prompt(
    step4_prompt,
    "CoVe Step 4: Final Verified Analysis",
    analysis=draft_analysis,
    verifications=verification_results_2
)

--- CoVe Step 1: Draft Analysis ---
PROMPT:
Human: Draft a short paragraph analyzing Innovate Corp's latest quarterly results based on these data points: 


- Company: Innovate Corp.
- Revenue: ₹150 Crores (an increase of 12% Year-over-Year).
- Net Profit: ₹25 Crores (an increase of 8% Year-over-Year).
- Key Highlight: Successfully launched new AI platform 'Cognito'.
- Stock Price Movement (last quarter): -5%.
- Analyst Consensus Rating: 'Hold'.


==================== RESPONSE ====================



Innovate Corp.'s latest quarterly results reflect steady growth, with revenue rising 12% year-over-year to ₹150 Crores and net profit increasing by 8% to ₹25 Crores. A significant highlight was the successful launch of their new AI platform, 'Cognito', which positions the company well for future innovation-driven growth. However, despite these positive financial metrics, the stock price declined by 5% over the quarter, indicating some market caution. This sentiment is echoed by analysts, who have maintained a consensus rating of 'Hold', suggesting that while the company shows promise, investors may be awaiting further evidence of sustained momentum.


--------------------------------------------------

--- CoVe Step 2: Plan Verifications ---
PROMPT:
Human: 
Based on the following draft analysis, generate a list of verification questions to fact-check its claims against the original data points.

Draft Analysis:
Innovate Corp.'s latest quarterly results reflect steady growth, with revenue rising 12% year-over-year to ₹150 Crores and net profit increasing by 8% to ₹25 Crores. A significant highlight was the successful launch of their new AI platform, 'Cognito', which positions the company well for future innovation-driven growth. However, despite these positive financial metrics, the stock price declined by 5% over the quarter, indicating some market caution. This sentiment is echoed by analysts, who have maintained a consensus rating of 'Hold', suggesting that while the company shows promise, investors may be awaiting further evidence of sustained momentum.


==================== RESPONSE ====================



Here is a list of verification questions to fact-check the draft analysis against the original data points:

1. What was Innovate Corp.'s reported revenue for the latest quarter, and how does it compare year-over-year? Is the stated 12% increase to ₹150 Crores accurate?
2. What was the net profit reported for the latest quarter, and does it reflect an 8% increase to ₹25 Crores as claimed?
3. Was the AI platform 'Cognito' officially launched during the latest quarter? Are there details confirming its launch and significance?
4. Did Innovate Corp.'s stock price decline by 5% over the latest quarter? What is the exact percentage change in the stock price during this period?
5. What is the current consensus analyst rating for Innovate Corp.? Is it indeed a 'Hold' rating, and what are the reasons provided by analysts for this stance?
6. Are there any statements or data points indicating that the company is positioned for future innovation-driven growth due to the launch of 'Cognito'?
7. Is there evidence of market caution or investor sentiment that aligns with the reported stock price decline and analyst rating?
8. Are the financial metrics (revenue and net profit) consistent with the official quarterly financial statements released by Innovate Corp.?
9. Were there any other significant events or factors during the quarter that might have influenced the stock price or analyst ratings not mentioned in the draft?


--------------------------------------------------

--- CoVe Step 3: Execute Verifications ---
PROMPT:
Human: 
Answer the following questions based *only* on the provided data points. 
Do not infer or speculate.

Data Points:

- Company: Innovate Corp.
- Revenue: ₹150 Crores (an increase of 12% Year-over-Year).
- Net Profit: ₹25 Crores (an increase of 8% Year-over-Year).
- Key Highlight: Successfully launched new AI platform 'Cognito'.
- Stock Price Movement (last quarter): -5%.
- Analyst Consensus Rating: 'Hold'.


Questions:
Here is a list of verification questions to fact-check the draft analysis against the original data points:

1. What was Innovate Corp.'s reported revenue for the latest quarter, and how does it compare year-over-year? Is the stated 12% increase to ₹150 Crores accurate?
2. What was the net profit reported for the latest quarter, and does it reflect an 8% increase to ₹25 Crores as claimed?
3. Was the AI platform 'Cognito' officially launched during the latest qua

Based *only* on the provided data points, here are the answers to the verification questions:

1. **Revenue and Year-over-Year Increase:**  
   Innovate Corp.'s reported revenue for the latest quarter is ₹150 Crores, which reflects a 12% Year-over-Year increase. The stated 12% increase to ₹150 Crores is accurate based on the data provided.

2. **Net Profit and Year-over-Year Increase:**  
   The net profit reported for the latest quarter is ₹25 Crores, showing an 8% Year-over-Year increase. This matches the claimed figures exactly.

3. **Launch of AI Platform 'Cognito':**  
   The data confirms that Innovate Corp. successfully launched the new AI platform 'Cognito'. There is no additional detail provided about the timing beyond "successfully launched" or the significance beyond it being a key highlight.

4. **Stock Price Movement:**  
   Innovate Corp.'s stock price declined by 5% over the latest quarter. The exact percentage change provided is -5%.

5. **Consensus Analyst Rating:**  
   The current consensus analyst rating for Innovate Corp. is 'Hold'. No reasons or explanations for this rating are provided in the data.

6. **Statements on Future Innovation-Driven Growth:**  
   There are no statements or data points indicating that the company is positioned for future innovation-driven growth due to the launch of 'Cognito' beyond it being listed as a key highlight.

7. **Market Caution or Investor Sentiment:**  
   There is no explicit evidence or statements regarding market caution or investor sentiment that align with the stock price decline and analyst rating in the provided data.

8. **Consistency of Financial Metrics with Official Statements:**  
   The financial metrics (₹150 Crores revenue with 12% increase, ₹25 Crores net profit with 8% increase) are consistent with the official quarterly financial data points given.

9. **Other Significant Events or Factors:**  
   No other significant events or factors during the quarter that might have influenced the stock price or analyst ratings are mentioned in the data.

---

**Summary:**  
All factual figures and key points in the draft analysis regarding revenue, net profit, AI platform launch, stock price movement, and analyst rating are accurate as per the provided data. However, no additional context, reasons, or qualitative insights beyond these data points are available.


--------------------------------------------------

--- CoVe Step 4: Final Verified Analysis ---
PROMPT:
Human: 
You have a draft analysis and a set of verified facts.
Your task is to rewrite the analysis, ensuring every statement is directly supported by the verified facts.
Remove any speculation or causal links that are not explicitly mentioned in the data.

Draft Analysis:
Innovate Corp.'s latest quarterly results reflect steady growth, with revenue rising 12% year-over-year to ₹150 Crores and net profit increasing by 8% to ₹25 Crores. A significant highlight was the successful launch of their new AI platform, 'Cognito', which positions the company well for future innovation-driven growth. However, despite these positive financial metrics, the stock price declined by 5% over the quarter, indicating some market caution. This sentiment is echoed by analysts, who have maintained a consensus rating of 'Hold', suggesting that while the company shows promise, investors may be awaiting fu

Innovate Corp.'s latest quarterly results show revenue of ₹150 Crores, reflecting a 12% year-over-year increase, and net profit of ₹25 Crores, an 8% year-over-year increase. The company successfully launched its new AI platform, 'Cognito', during the quarter. Despite these financial results, the stock price declined by 5% over the same period. The consensus analyst rating for Innovate Corp. remains at 'Hold'.


--------------------------------------------------



This final version is factually impeccable. It presents the different data points without inventing a narrative to connect them, which is exactly what's required for unbiased financial reporting.

---
## 9. Conclusion & Key Takeaways

### What We Covered

| Concept | Takeaway |
|---|---|
| **Zero-shot vs few-shot** | Zero-shot for simple tasks on capable models; few-shot when format is unusual or accuracy matters |
| **Chain-of-Thought (CoT)** | Verbalize reasoning before answer — auditable, accurate, slower |
| **Tree-of-Thought (ToT)** | Explore multiple hypotheses in parallel and pick the best — for ambiguous, multi-strategy decisions |
| **Chain-of-Verification (CoVe)** | Draft → plan questions → answer independently → revise — substantial hallucination reduction |
| **Reasoning models change the math** | On `gpt-5-mini` / `o1` you don't need 'let's think step by step' — internal CoT happens automatically |
| **Cost / latency trade-off** | CoT is ~2× longer output; CoVe is ~3-4× total tokens. Worth it when accuracy matters more than speed |

**Next Lab:** Lab 2.4 — LangChain Output Parsers (PydanticOutputParser, XML, structured output) 📦


## 10. Stretch Exercise (Optional)

1. Take the loan-application CoT example and rerun it on `gpt-5-mini` with `reasoning_effort='medium'` — compare quality and total tokens.
2. Implement a 5-thought ToT loop that *programmatically* enumerates options, scores each with a separate LLM call, and returns the top scorer. Compare to the single-prompt ToT-style version.
3. Build a 10-document factual benchmark and measure CoVe's hallucination-reduction empirically (LLM-as-Judge for groundedness).
4. Add a 'CoVe-on-CoT' compound: verify each step of a CoT reasoning chain, not just the final answer.
5. Wire the CoVe loop into a LangGraph state machine — preview of Module 7.


---

## Interview Preparation

The questions below mirror what client interviewers commonly ask about the topics in this lab. Use the hint to think through the answer first; use the sketch only to verify your reasoning.

---

**Q1. When does Chain-of-Thought help, and when does it hurt?**

*Hint:* Think latency, cost, and whether the task actually benefits from explicit reasoning.

*Answer sketch:* CoT helps for multi-step problems (math, legal reasoning, multi-criteria decisions) — measurably better accuracy. It hurts when latency/cost matter and the task is simple (classification, extraction): you pay 2-3× output tokens for no quality gain. On reasoning models like `gpt-5-mini`, manual CoT is also redundant — the model thinks internally.

---

**Q2. Difference between zero-shot, few-shot, and CoT — when to choose each?**

*Hint:* Examples vs reasoning vs neither.

*Answer sketch:* Zero-shot = just describe the task — works on capable models for common tasks. Few-shot = 2-8 input/output examples — locks unusual formats and decision boundaries. CoT = ask the model to reason step-by-step — unlocks multi-step accuracy. Stack them: few-shot CoT (examples that *show* reasoning) is the strongest pattern for hard tasks on non-reasoning models.

---

**Q3. What does Tree-of-Thought (ToT) actually mean — what is the algorithm?**

*Hint:* Beyond a linear chain of reasoning.

*Answer sketch:* ToT generates *multiple* candidate reasoning paths in parallel, evaluates each (often via a separate LLM call), and selects/expands the best. The full algorithm is search-tree-based with backtracking; the prompt-only 'ToT-style' version (this lab) just asks the model to enumerate and compare options in one shot. The full version trades cost for robustness on ambiguous decisions.

---

**Q4. Why does CoVe reduce hallucination — what's the underlying mechanism?**

*Hint:* Draft → verify → revise.

*Answer sketch:* By forcing the model to (a) plan verification questions and (b) answer each *independently* using only the source, CoVe breaks the autoregressive momentum that often produces fluent-but-wrong claims. Each verification answer is a fresh prompt with focused context, so unsupported claims don't survive the second pass.

---

**Q5. When would you skip CoVe despite the accuracy gain?**

*Hint:* Latency, cost, and use-case stakes.

*Answer sketch:* Skip when latency matters (CoVe = 3-4× round trips), when output is non-factual (creative writing), or when stakes are low (drafting boilerplate). Use it for compliance reports, regulatory summaries, financial analysis, customer-facing facts — places where a hallucination has real-world cost.

---

**Q6. How do you decide how many few-shot examples to provide?**

*Hint:* Diminishing returns vs context cost.

*Answer sketch:* Start at 2-3 (covers the common cases). Add examples that demonstrate *edge cases* the model is mishandling — not more of the same. Marginal value drops sharply after ~8 examples. Above that, switch to dynamic example selection (semantic similarity to the input) rather than adding more static examples.

---

**Q7. What's the role of 'let's think step by step' — is it still effective on newer models?**

*Hint:* Reasoning models do this internally.

*Answer sketch:* On older / smaller / non-reasoning models (Llama 3, Claude Haiku, gpt-4.1-mini), the phrase still triggers explicit CoT and improves multi-step accuracy. On reasoning models (`gpt-5-mini`, `o1`, Claude with extended thinking), it's redundant — they think internally regardless. Setting `reasoning_effort` is the right knob there.

